# Notebook 5: SCIGEN — Constrained Crystal Generation (Capstone)

**APS Tutorial T4 — Generative AI for Physics: From Models to Materials**

This is the capstone notebook. Everything from the previous sessions comes together here:
- **Notebook 01:** We learned to represent crystal structures with pymatgen
- **Notebook 02:** We explored the Materials Project and saw that kagome materials are rare
- **Notebook 03:** We learned how diffusion models generate crystals
- **Notebook 04:** We learned to evaluate structures with MLIPs

Now we will **generate new crystal structures** with targeted geometric constraints
using the SCIGEN diffusion model.

**Paper:** [Structural constraint integration in a generative model for the discovery of quantum materials](https://doi.org/10.1038/s41563-025-02355-y) (Nature Materials, 2025)

> **Prerequisites:** Run `00_setup.ipynb` first (installs dependencies and downloads model weights).

In [ ]:
# --- Run once per Colab session (skip if you already ran 00_setup.ipynb) ---
import os, shutil, subprocess, sys

REPO_URL = 'https://github.com/RyotaroOKabe/APS_demo_SCIGEN.git'
PROJECT_DIR = '/content/APS_demo_SCIGEN'

# Clone repo if needed
if not os.path.exists(os.path.join(PROJECT_DIR, '.git')):
    if os.path.exists(PROJECT_DIR):
        shutil.rmtree(PROJECT_DIR)
    os.system(f'git clone {REPO_URL} {PROJECT_DIR}')

# Install PyG extensions (need matching torch+CUDA wheels)
try:
    import torch_scatter
except ImportError:
    import torch
    _vp = torch.__version__.split('+')[0].split('.')
    tv = f'{_vp[0]}.{_vp[1]}.0'
    if torch.version.cuda:
        cv = 'cu' + torch.version.cuda.replace('.', '')
    else:
        cv = 'cpu'
    whl = f'https://data.pyg.org/whl/torch-{tv}+{cv}.html'
    print(f'Installing PyG extensions from: {whl}')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                           'torch-scatter', 'torch-sparse', 'torch-cluster',
                           '-f', whl])

# Install all tutorial dependencies from the shared requirements file
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
                       '-r', f'{PROJECT_DIR}/requirements-colab.txt'])

# Download model weights if not present
from pathlib import Path
MODEL_PATH = Path(PROJECT_DIR) / 'models' / 'mp_20'
if not MODEL_PATH.exists() or not list(MODEL_PATH.glob('*.ckpt')):
    import json, zipfile
    from urllib.request import urlopen, urlretrieve
    MODEL_PATH.mkdir(parents=True, exist_ok=True)
    article = json.loads(urlopen('https://api.figshare.com/v2/articles/27778134').read().decode())
    for f in article.get('files', []):
        dest = MODEL_PATH / f['name']
        print(f'Downloading {f["name"]}...')
        urlretrieve(f['download_url'], str(dest))
        if dest.suffix == '.zip':
            with zipfile.ZipFile(dest, 'r') as zf:
                zf.extractall(MODEL_PATH)
            dest.unlink()

os.environ.setdefault('PROJECT_ROOT', PROJECT_DIR)
print('Ready.')

---
## 5.1 What is SCIGEN?

SCIGEN (Structural Constraint Integration in GENerative model) is a diffusion-based
generative model for crystal structure discovery. Its key innovation:

**You specify a geometric constraint** (kagome, honeycomb, triangular, etc.)
**and SCIGEN generates complete crystal structures** that contain that sublattice.

The model uses an **inpainting** approach:
1. The diffusion model generates the full structure (all atoms) from noise
2. The constrained atom positions (defining the desired geometric pattern) are **inpainted** — revealed at the end, replacing the model's predictions at those sites
3. During training, the model learns to anticipate these constraints and generate complementary atoms that form physically reasonable crystals around the target sublattice
4. The lattice geometry is also constrained (e.g., hexagonal cell for kagome)

In the published work, this pipeline generated 10 million candidate structures,
which were screened down to 24,743 DFT-validated materials.

---
## 5.2 Environment setup

If you ran `00_setup.ipynb`, the environment is already configured.
If running this notebook standalone, the cells below handle setup.

In [ ]:
import os, sys

PROJECT_DIR = os.environ.get('PROJECT_ROOT', '/content/APS_demo_SCIGEN')

# Set env vars if not already set (e.g., running standalone)
os.environ.setdefault('PROJECT_ROOT', PROJECT_DIR)
os.environ.setdefault('HYDRA_JOBS', PROJECT_DIR)
os.environ.setdefault('WANDB_DIR', os.path.join(PROJECT_DIR, 'wandb'))
os.makedirs(os.environ['WANDB_DIR'], exist_ok=True)

if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)
if os.path.join(PROJECT_DIR, 'script') not in sys.path:
    sys.path.insert(0, os.path.join(PROJECT_DIR, 'script'))

os.chdir(PROJECT_DIR)
print(f'Working directory: {os.getcwd()}')

---
## 5.3 Load the pretrained model

We load the SCIGEN diffusion model using Hydra for configuration and manual
state-dict loading for compatibility with any PyTorch Lightning version.

In [ ]:
import torch
import numpy as np
import hydra
from hydra import compose, initialize_config_dir
from hydra.core.global_hydra import GlobalHydra
from pathlib import Path

import scigen
print(f'scigen imported from: {scigen.__path__[0]}')


def load_model_for_inference(model_path, device='cpu'):
    """Load SCIGEN pretrained model for inference.

    Uses manual state_dict loading instead of pytorch_lightning's
    load_from_checkpoint, so it works with any PL version.
    """
    model_path = Path(model_path)

    # Clear previous hydra state (allows re-running this cell)
    GlobalHydra.instance().clear()

    # Load model config from hparams.yaml
    with initialize_config_dir(config_dir=str(model_path.resolve()), version_base=None):
        cfg = compose(config_name='hparams')

    # Instantiate model architecture (empty weights)
    model = hydra.utils.instantiate(
        cfg.model, optim=cfg.optim, data=cfg.data,
        logging=cfg.logging, _recursive_=False,
    )

    # Find checkpoint file
    ckpts = sorted(model_path.glob('*.ckpt'))
    if not ckpts:
        raise FileNotFoundError(f'No .ckpt files found in {model_path}')

    ckpt_path = next((c for c in ckpts if 'last' in c.name), ckpts[-1])
    print(f'Loading checkpoint: {ckpt_path.name}')

    checkpoint = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    model.load_state_dict(checkpoint['state_dict'], strict=False)

    # Load data scalers
    for attr, fname in [('lattice_scaler', 'lattice_scaler.pt'),
                        ('scaler', 'prop_scaler.pt')]:
        fpath = model_path / fname
        if fpath.exists():
            setattr(model, attr, torch.load(fpath, map_location='cpu', weights_only=False))

    model = model.to(device)
    model.eval()
    return model, cfg

In [ ]:
MODEL_PATH = Path(PROJECT_DIR) / 'models' / 'mp_20'
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

model, cfg = load_model_for_inference(MODEL_PATH, device=device)

# Attach the generation sampling method
from scigen.pl_modules.diffusion_w_type import sample_scigen
model.sample_scigen = sample_scigen.__get__(model)

num_params = sum(p.numel() for p in model.parameters())
print(f'Model loaded: {num_params:,} parameters')

---
## 5.4 Choose your structural constraint

This is the key feature of SCIGEN: you pick a lattice geometry, and the model
generates complete crystal structures containing that sublattice.

| Code | Lattice | Known atoms | Description |
|------|---------|-------------|-------------|
| `kag` | **Kagome** | 3 | Corner-sharing triangles — frustrated magnets |
| `hon` | **Honeycomb** | 2 | Graphene-like — topological materials |
| `tri` | **Triangular** | 1 | Simplest 2D lattice |
| `sqr` | **Square** | 1 | Square lattice |
| `lieb` | **Lieb** | 3 | Square with flat bands |
| `elt` | Elongated triangular | 2 | Rectangular + triangular |
| `sns` | Snub square | 4 | Archimedean tiling |
| `tsq` | Truncated square | 4 | Squares + octagons |
| `srt` | Small rhombotrihexagonal | 6 | Complex tiling |
| `snh` | Snub hexagonal | 6 | Complex tiling |
| `trh` | Truncated hexagonal | 6 | Complex tiling |
| `grt` | Great rhombotrihexagonal | 12 | Most complex tiling |
| `van` | Vanilla (no constraint) | 1 | Unconditional generation |

In [ ]:
from torch_geometric.data import DataLoader
from gen_utils import SampleDataset

# ============================================================
#  EDIT THESE PARAMETERS to try different generation settings
# ============================================================
SC_TYPE    = 'kag'    # Structural constraint (see table above)
ELEMENT    = 'Mn'     # Element for constrained sites
BATCH_SIZE = 4        # Structures per batch (keep small for Colab)
NUM_BATCHES = 1       # Number of batches
# ============================================================

FRAC_Z   = 0.5        # z-coordinate of the 2D pattern layer
STEP_LR  = 5e-6       # Diffusion step size
SEED     = 42

# Atom count ranges per structural constraint
SC_NATM_RANGE = {
    'tri': [1, 4],  'hon': [1, 8],  'kag': [1, 12],
    'sqr': [1, 4],  'elt': [1, 8],  'sns': [1, 16],
    'tsq': [1, 16], 'van': [1, 20], 'lieb': [1, 12],
    'srt': [1, 20], 'snh': [1, 20], 'trh': [1, 20],
    'grt': [1, 20],
}

total_structures = BATCH_SIZE * NUM_BATCHES
natm_range = SC_NATM_RANGE.get(SC_TYPE, [1, 20])

print(f'Generation settings:')
print(f'  Lattice type:      {SC_TYPE}')
print(f'  Element:           {ELEMENT}')
print(f'  Structures:        {total_structures}')
print(f'  Atoms per cell:    {natm_range[0]}-{natm_range[1]}')

---
## 5.5 Run the diffusion model

This is where the magic happens. The model starts from random noise and
gradually denoises to produce crystal structures. The constrained atom positions
are inpainted at the end to enforce the target lattice geometry.

In [ ]:
import time

# Build the sample dataset with structural constraints
test_set = SampleDataset(
    dataset='mp_20',
    natm_range=natm_range,
    total_num=total_structures,
    bond_sigma_per_mu=None,
    use_min_bond_len=False,
    known_species=[ELEMENT],
    sc_list=[SC_TYPE],
    frac_z=FRAC_Z,
    c_vec_cons={'scale': None, 'vert': False},
    reduced_mask=False,
    seed=SEED,
    device=device,
)

test_loader = DataLoader(test_set, batch_size=BATCH_SIZE)

# Run diffusion sampling
all_frac_coords = []
all_atom_types = []
all_lattices = []
all_num_atoms = []
all_num_known = []

print(f'Running diffusion (step_lr={STEP_LR})...')
start_time = time.time()

for idx, batch in enumerate(test_loader):
    if torch.cuda.is_available():
        batch.cuda()
    outputs, traj = model.sample_scigen(batch, step_lr=STEP_LR)

    all_frac_coords.append(outputs['frac_coords'].detach().cpu())
    raw_types = outputs['atom_types'].detach().cpu()
    if raw_types.dim() == 2:
        raw_types = raw_types.argmax(dim=-1) + 1
    all_atom_types.append(raw_types)
    all_lattices.append(outputs['lattices'].detach().cpu())
    all_num_atoms.append(outputs['num_atoms'].detach().cpu())
    all_num_known.append(outputs['num_known'].detach().cpu())

    print(f'  Batch {idx + 1}/{len(test_loader)} complete')

elapsed = time.time() - start_time
print(f'\nGeneration complete in {elapsed:.1f}s ({elapsed / total_structures:.1f}s per structure)')

# Concatenate results
frac_coords = torch.cat(all_frac_coords, dim=0)
atom_types = torch.cat(all_atom_types, dim=0)
lattices = torch.cat(all_lattices, dim=0)
num_atoms = torch.cat(all_num_atoms, dim=0)
num_known = torch.cat(all_num_known, dim=0)

---
## 5.6 Inspect the generated structures

Let's convert the raw tensor outputs to pymatgen Structure objects and examine
what the model produced.

In [ ]:
def lattices_to_params(lattices):
    """Convert 3x3 lattice matrices to (lengths, angles) parameterization."""
    lengths = torch.sqrt(torch.sum(lattices ** 2, dim=-1))
    angles = torch.zeros_like(lengths)
    for i in range(3):
        j, k = (i + 1) % 3, (i + 2) % 3
        cos_angle = torch.clamp(
            torch.sum(lattices[..., j, :] * lattices[..., k, :], dim=-1)
            / (lengths[..., j] * lengths[..., k]),
            -1.0, 1.0,
        )
        angles[..., i] = torch.arccos(cos_angle) * 180.0 / np.pi
    return lengths, angles


lengths, angles = lattices_to_params(lattices)
num_known_flat = num_known.flatten()

from scigen.common.data_utils import chemical_symbols
from collections import Counter

print(f'Generated {num_atoms.shape[0]} structures\n')
print(f'{"#":>3} {"Atoms":>5} {"Known":>5} {"Composition":<25} {"a":>7} {"b":>7} {"c":>7}  {"α":>6} {"β":>6} {"γ":>6}')
print('-' * 95)

start_idx = 0
for i in range(num_atoms.shape[0]):
    na = int(num_atoms[i])
    nk = int(num_known_flat[i]) if i < len(num_known_flat) else 0
    types_i = atom_types[start_idx:start_idx + na].int().tolist()
    symbols = [chemical_symbols[t] for t in types_i]
    comp = Counter(symbols)
    comp_str = ' '.join(f'{el}{n}' for el, n in sorted(comp.items()))
    l = lengths[i].tolist()
    a = angles[i].tolist()
    print(f'{i:3d} {na:5d} {nk:5d} {comp_str:<25} {l[0]:7.2f} {l[1]:7.2f} {l[2]:7.2f}  {a[0]:6.1f} {a[1]:6.1f} {a[2]:6.1f}')
    start_idx += na

In [ ]:
# Convert to pymatgen Structure objects
from pymatgen.core.structure import Structure
from pymatgen.core.lattice import Lattice

structures = []
start_idx = 0

for i in range(num_atoms.shape[0]):
    na = int(num_atoms[i])
    types_i = atom_types[start_idx:start_idx + na].int().tolist()
    coords_i = frac_coords[start_idx:start_idx + na].numpy()
    lengths_i = lengths[i].tolist()
    angles_i = angles[i].tolist()

    species = [chemical_symbols[t] for t in types_i]
    lattice = Lattice.from_parameters(*lengths_i, *angles_i)

    try:
        structure = Structure(lattice, species, coords_i, coords_are_cartesian=False)
        structures.append(structure)
        print(f'Structure {i}: {structure.composition.reduced_formula}'
              f'  ({na} atoms, V={structure.volume:.1f} Å³)')
    except Exception as e:
        print(f'Structure {i}: construction failed - {e}')

    start_idx += na

print(f'\nSuccessfully constructed {len(structures)}/{num_atoms.shape[0]} structures')

---
## 5.7 Visualize the generated structures

Let's see what the diffusion model produced.

In [ ]:
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D


def plot_structure_3d(structure, title='', ax=None):
    """3D scatter plot of atom positions in a crystal structure."""
    if ax is None:
        fig = plt.figure(figsize=(6, 5))
        ax = fig.add_subplot(111, projection='3d')

    cart_coords = structure.cart_coords
    species = [str(s.specie) for s in structure]

    unique_el = sorted(set(species))
    cmap = plt.cm.Set1(np.linspace(0, 0.9, max(len(unique_el), 1)))
    color_map = {el: cmap[i] for i, el in enumerate(unique_el)}

    for el in unique_el:
        mask = [s == el for s in species]
        coords = cart_coords[mask]
        ax.scatter(
            coords[:, 0], coords[:, 1], coords[:, 2],
            s=200, c=[color_map[el]], label=el,
            alpha=0.8, edgecolors='k', linewidths=0.5,
        )

    ax.set_xlabel('x (Å)')
    ax.set_ylabel('y (Å)')
    ax.set_zlabel('z (Å)')
    ax.legend(loc='upper left', fontsize=8)
    ax.set_title(title, fontsize=10)
    return ax


n_plot = min(4, len(structures))
fig = plt.figure(figsize=(5 * n_plot, 5))

for i in range(n_plot):
    ax = fig.add_subplot(1, n_plot, i + 1, projection='3d')
    formula = structures[i].composition.reduced_formula
    plot_structure_3d(structures[i], title=f'#{i}: {formula}', ax=ax)

plt.tight_layout()
plt.show()

---
## 5.8 Export as CIF files

Save the generated structures as CIF files for further analysis
(e.g., DFT relaxation, MLIP evaluation from Notebook 04).

In [ ]:
output_dir = os.path.join(PROJECT_DIR, 'generated_cifs')
os.makedirs(output_dir, exist_ok=True)

for i, structure in enumerate(structures):
    formula = structure.composition.reduced_formula
    cif_path = os.path.join(output_dir, f'{SC_TYPE}_{formula}_{i:03d}.cif')
    structure.to(filename=cif_path, fmt='cif')
    print(f'Saved: {os.path.basename(cif_path)}')

print(f'\n{len(structures)} CIF files saved to {output_dir}')

In [ ]:
# Download CIF files as a zip archive (Colab only)
try:
    import shutil
    from google.colab import files
    zip_name = f'scigen_{SC_TYPE}_{ELEMENT}'
    zip_path = shutil.make_archive(
        os.path.join(PROJECT_DIR, zip_name), 'zip', output_dir
    )
    files.download(zip_path)
    print(f'Downloading {zip_name}.zip')
except ImportError:
    print('Not running in Colab — skipping download.')
except Exception as e:
    print(f'Download failed: {e}')
    print(f'CIF files are available at: {output_dir}')

---
## 5.9 (Optional) Evaluate with CHGNet

If you completed Notebook 04, you can evaluate the generated structures
with CHGNet right here.

In [ ]:
try:
    from chgnet.model.model import CHGNet
    chgnet = CHGNet.load()

    print(f'{"#":>3} {"Formula":<20} {"Energy (eV/atom)":>18} {"Max |F| (eV/Å)":>16}')
    print('-' * 62)
    for i, s in enumerate(structures):
        pred = chgnet.predict_structure(s)
        formula = s.composition.reduced_formula
        print(f'{i:3d} {formula:<20} {float(pred["e"]):18.4f} {abs(pred["f"]).max():16.4f}')

except ImportError:
    print('CHGNet not installed. Run 00_setup.ipynb or: pip install chgnet')
except Exception as e:
    print(f'CHGNet evaluation failed: {e}')

---
## 5.10 Try it yourself!

Go back to **Section 5.4** and change the parameters:

- **Different lattice:** Change `SC_TYPE` to `'hon'` (honeycomb), `'tri'` (triangular), `'sqr'` (square), etc.
- **Different element:** Change `ELEMENT` to `'Fe'`, `'Co'`, `'Ni'`, `'Ti'`, etc.
- **More structures:** Increase `BATCH_SIZE` or `NUM_BATCHES`
- **Custom constraints:** See the [README](https://github.com/RyotaroOKabe/APS_demo_SCIGEN#create-your-own-structural-constraint) for defining your own lattice types

Then re-run from Section 5.5 onward.

---
## Summary

In this tutorial, we:
1. **Represented** crystal structures with pymatgen (Notebook 01)
2. **Explored** materials databases and motivated the need for generative models (Notebook 02)
3. **Understood** how diffusion models generate crystals (Notebook 03)
4. **Evaluated** structures with machine learning potentials (Notebook 04)
5. **Generated** new crystal structures with SCIGEN's structural constraints (this notebook)

### The full SCIGEN pipeline

```
Choose constraint (kagome, honeycomb, ...)
    → Generate candidates with SCIGEN
    → Pre-screen (composition, bond distances)
    → MLIP evaluation (CHGNet, GNN models)
    → DFT validation of best candidates
    → Experimental synthesis
```

For more details, see the [SCIGEN paper](https://doi.org/10.1038/s41563-025-02355-y) (Nature Materials, 2025).